# 15 · Prospective operations, freshness gates and performance monitoring

**Purpose:** Finish the housekeeping before Notebook 16 (ensembles and neural networks from scratch). No new classifiers are trained here. Use the frozen Notebook 14 logistic L2 artifact.

**Sequence:** Read current database → inspect source → load frozen model → reconcile prior predictions → review new predictions → *explicitly confirm* completed session → append → save run reports. The final historical holdout is closed. No backfilled predictions are called prospective.

**Important:** Daily database timestamps may be session-date labels (e.g. midnight UTC), not actual market-close times. This notebook cannot automatically certify that a market session is final. You must confirm ingestion completeness and the exchange session date before committing.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from stock_ml_dynamic.db import engine
from stock_ml_dynamic.master_table import fetch_universe_prices
from stock_ml_dynamic.prospective import (prepare_universe,load_model,predict_latest,append_predictions,reconcile_predictions)
from stock_ml_dynamic.prospective_operations import (source_audit,recording_gate,ledger_audit,evaluation_report,save_operational_report)
INTERVAL='1d'
OUT=ROOT/'artifacts'/'prospective'/INTERVAL
MODELS=OUT/'models'
LEDGER=OUT/'prediction_ledger.parquet'
# Keep the FIRST frozen version for a clean prospective evaluation.
MODEL_PATH=None  # Optionally set explicit trusted path to model.joblib
# IMPORTANT: default is read-only preview. Change only after verifying the completed session.
COMMIT_NEW_PREDICTIONS=False
OPERATOR_CONFIRMED=False
CONFIRMED_SESSION_DATE=None  # e.g. '2026-11-10' ONLY when actually confirmed complete
MIN_COVERAGE=0.95
MAX_SOURCE_LAG_DAYS=4  # Informational; exchange holidays may exceed this.


## 1 · Fetch and audit current OHLCV data

In [ ]:
with engine().connect() as conn:
    raw_prices=fetch_universe_prices(conn,interval=INTERVAL,bars_per_symbol=2000)
source_by_symbol,source_report=source_audit(raw_prices,interval=INTERVAL,max_lag_days=MAX_SOURCE_LAG_DAYS)
print(json.dumps(source_report,indent=2))
display(source_by_symbol.loc[~source_by_symbol.eligible_history | source_by_symbol.stale | ~source_by_symbol.latest_is_modal | source_by_symbol.invalid_latest_price].head(40))
print('Latest bar counts:')
display(source_by_symbol.groupby('latest_bar').size().sort_index().tail(12))


## 2 · Load the existing frozen model (do not retrain)

Notebook 16 will benchmark new models separately. Notebook 15 must not silently update the deployed classifier. If no model exists, run Notebook 14 once to create it.

In [ ]:
if MODEL_PATH is None:
    candidates=sorted(MODELS.glob('*/model.joblib'))
    if not candidates:
        raise FileNotFoundError(f'No frozen model in {MODELS}; create one with Notebook 14 first')
    model_path=candidates[0]  # oldest frozen version
else:
    model_path=Path(MODEL_PATH).expanduser().resolve()
bundle=load_model(model_path)
print('Frozen model:',model_path)
print('Version:',bundle['model_version'],'trained through:',bundle['training_end'])
frame,extra,eligibility=prepare_universe(raw_prices)
print('Eligible:',frame.symbol.nunique(),'of',eligibility.symbol.nunique())
display(eligibility.loc[~eligibility.eligible].head(20))


## 3 · Reconcile previously recorded predictions

This step updates **only realized outcome fields**. Existing probabilities and predictions are not retrained or replaced. A later source bar is not necessarily a completed bar; confirm ingestion before running reconciliation.

In [ ]:
if LEDGER.exists():
    ledger, resolved_now=reconcile_predictions(LEDGER,raw_prices)
    print('Newly reconciled:',resolved_now)
else:
    ledger=pd.DataFrame()
    print('No existing ledger; no prospective scores yet.')
ledger_summary,by_version=ledger_audit(ledger)
print('Ledger audit:',ledger_summary)
display(by_version)
scores,per_class=evaluation_report(ledger)
display(scores)
display(per_class)


## 4 · Preview new next-interval predictions

Review the session date, number of eligible symbols, and latest price bars. Predictions must be recorded **before the next outcome exists**. This cell does not write to the ledger.

In [ ]:
predictions=predict_latest(frame,bundle)
print('Predictions:',len(predictions),'symbols:',predictions.symbol.nunique())
print('Prediction as-of dates:',predictions.bar_timestamp.dt.date.value_counts().sort_index().to_dict())
display(predictions[['symbol','bar_timestamp','predicted_direction','prob_0','prob_1','prob_2']].head(15))


## 5 · Confirm and commit (explicit opt-in)

**Operator checklist:** ingestion finished; OHLCV bars final; confirmed session date matches source date labels; no newer observed bars; sufficient coverage; no simultaneous writer. `OPERATOR_CONFIRMED=True` is an affirmative statement, not an automated calendar check. If the source is stale or incomplete, leave commit disabled.

A saved prediction should be created *as soon as possible after the confirmed session closes*, not days later. For strict prospective evaluation, don't commit stale historical sessions.

In [ ]:
gate_report=None
if COMMIT_NEW_PREDICTIONS:
    gate_report=recording_gate(predictions,raw_prices,bundle,
        confirmed_session_date=CONFIRMED_SESSION_DATE,
        operator_confirmed=OPERATOR_CONFIRMED,
        min_coverage=MIN_COVERAGE)
    print('Approved gate:',gate_report)
    inserted,total=append_predictions(predictions,LEDGER)
    print('Inserted:',inserted,'total:',total)
    ledger=pd.read_parquet(LEDGER)
else:
    print('PREVIEW ONLY. No new predictions written. Set COMMIT_NEW_PREDICTIONS=True after review.')


## 6 · Write monitoring reports and ROC curves (resolved outcomes only)

The report is version-specific; it never mixes model versions into one claimed score. AUC is calculated only when all three classes are present and enough outcomes have resolved.

In [ ]:
ledger_summary,by_version=ledger_audit(ledger)
scores,per_class=evaluation_report(ledger,min_rows=30)
display(scores)
display(per_class)
report_dir=save_operational_report(OUT,source_report=source_report,
    gate_report=gate_report,ledger_report=ledger_summary,scores=scores,per_class=per_class)
print('Saved monitoring snapshot:',report_dir)
if not ledger.empty:
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve,auc
    resolved=ledger.loc[ledger.actual_direction.notna()]
    for version,g in resolved.groupby('model_version'):
        if len(g)<30 or set(g.actual_direction.astype(int).unique())!={0,1,2}:
            print('Insufficient resolved outcomes for ROC curves:',version,len(g));continue
        fig,ax=plt.subplots(figsize=(7,5))
        for cls,label in [(0,'Middle'),(1,'Up'),(2,'Down')]:
            y=(g.actual_direction.astype(int).to_numpy()==cls).astype(int)
            fpr,tpr,_=roc_curve(y,g[f'prob_{cls}'].to_numpy(dtype=float))
            ax.plot(fpr,tpr,label=f'{label} (AUC {auc(fpr,tpr):.3f})')
        ax.plot([0,1],[0,1],'--',color='gray',label='Random')
        ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f'Prospective ROC — {version}')
        ax.legend();fig.tight_layout()
        fig.savefig(report_dir/f'roc_{version}.png',dpi=150)
        plt.show()


## 7 · Handoff to Notebook 16

- Keep this logistic L2 model **frozen** as the prospective reference; don't overwrite its artifact or predictions.
- For ensembles and neural networks, use separate versioned model artifacts and separate prospective model-version keys.
- Use 5 chronological development folds and an untouched **new** evaluation period for model selection; the old final holdout has already been used.
- Measure wall-clock fit time, inference time, memory, class probabilities, ROC-AUC, balanced accuracy, and fold stability. For neural networks trained from scratch, begin with a reproducible PyTorch MLP, early stopping, and a deterministic seed.
- Notebook 15's ledger is **single-writer** and is not a production scheduler. For recurring automatic operation, add a controlled scheduler and transactional storage after verifying this workflow.